# Pipeline 1 - Pooled Assam, Kerala, Odisha Tabular Models

Consumes existing aligned tabular CSVs only. It does not regenerate preprocessing data, patch features, or U-Net outputs. The pooled Random Forest and XGBoost models retain the project held-out-year validation strategy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve)
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Paths and pooled-model settings. Change explicit state paths only if discovery cannot find them.
DRIVE_CANDIDATES = [Path('/content/drive/MyDrive/PRJ3_DATA'), Path('/content/drive/MyDrive/Prj_3_Data')]
DRIVE_ROOT = next((p for p in DRIVE_CANDIDATES if p.exists()), DRIVE_CANDIDATES[0])
OUTPUT_ROOT = DRIVE_ROOT / 'Pipeline1_Pooled_Tabular_Outputs'
MODEL_DIR = OUTPUT_ROOT / 'models'
REPORT_DIR = OUTPUT_ROOT / 'reports'
PLOT_DIR = OUTPUT_ROOT / 'plots'
for directory in [OUTPUT_ROOT, MODEL_DIR, REPORT_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

STATE_IDS = {'Assam': 0, 'Kerala': 1, 'Odisha': 2}
STATE_DATASET_PATHS = {'Assam': None, 'Kerala': None, 'Odisha': None}
TARGET = 'is_flood'
VAL_YEARS = [2023]  # Same full-year holdout strategy as the U-Net notebook.

def discover_aligned_dataset(state):
    candidates = sorted(DRIVE_ROOT.glob(f'**/{state.lower()}_aligned_tabular_dataset.csv'),
                        key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(
            f'No aligned CSV for {state}. Set STATE_DATASET_PATHS[{state!r}] to its existing CSV path.'
        )
    return candidates[0]

for state, path in STATE_DATASET_PATHS.items():
    path = Path(path) if path is not None else discover_aligned_dataset(state)
    if not path.exists():
        raise FileNotFoundError(path)
    STATE_DATASET_PATHS[state] = path
    print(f'{state}: {path}')


## 1. Merge aligned datasets and prepare the held-out-year split

The source manifest makes the combined CSV reusable only while all three source files are unchanged. Features are restricted to shared numeric columns; label-derived fields and row identifiers are excluded from model input.


In [ ]:
COMBINED_PATH = OUTPUT_ROOT / 'assam_kerala_odisha_aligned_tabular_dataset.csv'
SOURCE_MANIFEST_PATH = OUTPUT_ROOT / 'combined_source_manifest.csv'

def source_manifest(paths):
    return pd.DataFrame([{'state': state, 'path': str(path), 'bytes': path.stat().st_size,
                          'modified_ns': path.stat().st_mtime_ns} for state, path in paths.items()]).sort_values('state').reset_index(drop=True)

current_manifest = source_manifest(STATE_DATASET_PATHS)
reuse_combined = COMBINED_PATH.exists() and SOURCE_MANIFEST_PATH.exists() and pd.read_csv(SOURCE_MANIFEST_PATH).equals(current_manifest)

if reuse_combined:
    combined = pd.read_csv(COMBINED_PATH)
    print('Reused combined dataset:', COMBINED_PATH)
else:
    frames = []
    for state, path in STATE_DATASET_PATHS.items():
        frame = pd.read_csv(path)
        if TARGET not in frame.columns or 'year' not in frame.columns:
            raise ValueError(f'{path} must contain {TARGET!r} and year columns.')
        frame['state'] = state
        frame['state_id'] = np.int8(STATE_IDS[state])
        frames.append(frame)
    shared = set.intersection(*(set(frame.columns) for frame in frames))
    required = {'state', 'state_id', 'year', TARGET}
    if not required.issubset(shared):
        raise ValueError(f'State datasets do not share required fields: {sorted(required - shared)}')
    columns = [column for column in frames[0].columns if column in shared]
    combined = pd.concat([frame[columns] for frame in frames], ignore_index=True)
    combined.replace([np.inf, -np.inf], np.nan, inplace=True)
    for column in combined.select_dtypes(include=['float64']).columns:
        combined[column] = combined[column].astype(np.float32)
    combined.to_csv(COMBINED_PATH, index=False)
    current_manifest.to_csv(SOURCE_MANIFEST_PATH, index=False)
    del frames
    gc.collect()
    print('Saved combined dataset:', COMBINED_PATH)

combined[TARGET] = pd.to_numeric(combined[TARGET], errors='raise').astype(np.uint8)
combined['year'] = pd.to_numeric(combined['year'], errors='raise').astype(np.int16)
NON_FEATURES = {'state', 'year', 'chunk_id', 'local_patch', TARGET, 'label_flood_fraction', 'flood_fraction', 'is_empty_kept'}
feature_columns = [column for column in combined.columns if column not in NON_FEATURES and pd.api.types.is_numeric_dtype(combined[column])]
if 'state_id' not in feature_columns:
    feature_columns.append('state_id')
if not feature_columns:
    raise ValueError('No shared numeric features remain after removing labels and identifiers.')
combined[feature_columns] = combined[feature_columns].fillna(0).astype(np.float32)

val_mask = combined['year'].isin(VAL_YEARS).to_numpy()
if not val_mask.any() or val_mask.all():
    groups = combined['state'].astype(str) + '_' + combined['year'].astype(str)
    train_indices, val_indices = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(combined, groups=groups))
    train_mask = np.zeros(len(combined), dtype=bool)
    train_mask[train_indices] = True
    val_mask = ~train_mask
    split_strategy = 'grouped_state_year_fallback'
else:
    train_mask = ~val_mask
    split_strategy = f'holdout_years_{VAL_YEARS}'

X_train = combined.loc[train_mask, feature_columns].to_numpy(dtype=np.float32, copy=True)
X_val = combined.loc[val_mask, feature_columns].to_numpy(dtype=np.float32, copy=True)
y_train = combined.loc[train_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)
y_val = combined.loc[val_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)
if np.unique(y_train).size < 2 or np.unique(y_val).size < 2:
    raise ValueError('Both splits must contain flood and non-flood samples.')

split_summary = pd.DataFrame({'split': ['train', 'validation'], 'rows': [len(y_train), len(y_val)],
                              'flood_rate': [y_train.mean(), y_val.mean()], 'strategy': [split_strategy, split_strategy]})
split_summary.to_csv(REPORT_DIR / 'split_summary.csv', index=False)
combined.groupby(['state', 'year'])[TARGET].agg(rows='size', flood_rate='mean').reset_index().to_csv(REPORT_DIR / 'state_year_summary.csv', index=False)
print('Combined rows:', len(combined), '| shared numeric features:', len(feature_columns))
display(split_summary)


## 2. Train, resume, and evaluate pooled models

Saved models are reused when the feature signature is unchanged. Reruns always regenerate validation metrics, reports, prediction outputs, and plots from the saved models.


In [ ]:
try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('Run %pip install -q xgboost once in Colab, then rerun this cell.') from exc

RF_PATH = MODEL_DIR / 'pooled_random_forest.joblib'
XGB_PATH = MODEL_DIR / 'pooled_xgboost.joblib'
SIGNATURE_PATH = MODEL_DIR / 'feature_signature.csv'
signature = pd.DataFrame({'feature': feature_columns})
reuse_models = RF_PATH.exists() and XGB_PATH.exists() and SIGNATURE_PATH.exists() and pd.read_csv(SIGNATURE_PATH).equals(signature)

if reuse_models:
    rf_model, xgb_model = joblib.load(RF_PATH), joblib.load(XGB_PATH)
    print('Reused saved models.')
else:
    rf_model = RandomForestClassifier(n_estimators=400, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
    rf_model.fit(X_train, y_train)
    positive_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    xgb_model = XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0, objective='binary:logistic',
        eval_metric='logloss', tree_method='hist', max_bin=256, n_jobs=-1, random_state=SEED,
        scale_pos_weight=positive_weight)
    xgb_model.fit(X_train, y_train, verbose=False)
    joblib.dump(rf_model, RF_PATH, compress=3)
    joblib.dump(xgb_model, XGB_PATH, compress=3)
    signature.to_csv(SIGNATURE_PATH, index=False)

def evaluate_model(name, model):
    probability = model.predict_proba(X_val)[:, 1].astype(np.float32)
    prediction = (probability >= 0.5).astype(np.uint8)
    metrics = {'model': name, 'accuracy': accuracy_score(y_val, prediction),
               'precision': precision_score(y_val, prediction, zero_division=0),
               'recall': recall_score(y_val, prediction, zero_division=0),
               'f1': f1_score(y_val, prediction, zero_division=0),
               'roc_auc': roc_auc_score(y_val, probability),
               'average_precision': average_precision_score(y_val, probability)}
    report = classification_report(y_val, prediction, target_names=['non_flood', 'flood'], output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(REPORT_DIR / f'{name}_classification_report.csv')
    with open(REPORT_DIR / f'{name}_classification_report.txt', 'w') as handle:
        handle.write(classification_report(y_val, prediction, target_names=['non_flood', 'flood'], zero_division=0))
    id_columns = [c for c in ['state', 'state_id', 'year', 'chunk_id', 'local_patch'] if c in combined]
    predictions = combined.loc[val_mask, id_columns].copy()
    predictions['y_true'], predictions['probability'], predictions['prediction'] = y_val, probability, prediction
    predictions.to_csv(REPORT_DIR / f'{name}_validation_predictions.csv', index=False)

    cm = confusion_matrix(y_val, prediction)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, cmap='Blues')
    fig.colorbar(image, ax=ax)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['non-flood', 'flood'], yticklabels=['non-flood', 'flood'], xlabel='Predicted', ylabel='True', title=f'{name} confusion matrix')
    for row in range(2):
        for col in range(2): ax.text(col, row, str(cm[row, col]), ha='center', va='center')
    fig.tight_layout(); fig.savefig(PLOT_DIR / f'{name}_confusion_matrix.png', dpi=160); plt.show()

    fpr, tpr, _ = roc_curve(y_val, probability)
    precision, recall, _ = precision_recall_curve(y_val, probability)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.3f}"); axes[0].plot([0, 1], [0, 1], '--', color='gray'); axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'{name} ROC'); axes[0].legend()
    axes[1].plot(recall, precision, label=f"AP={metrics['average_precision']:.3f}"); axes[1].set(xlabel='Recall', ylabel='Precision', title=f'{name} precision-recall'); axes[1].legend()
    fig.tight_layout(); fig.savefig(PLOT_DIR / f'{name}_roc_pr.png', dpi=160); plt.show()

    importance = pd.DataFrame({'feature': feature_columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(REPORT_DIR / f'{name}_feature_importance.csv', index=False)
    top = importance.head(30).sort_values('importance')
    fig, ax = plt.subplots(figsize=(8, max(5, len(top) * 0.25)))
    ax.barh(top['feature'], top['importance'], color='#2a6f97'); ax.set(xlabel='Importance', title=f'{name} top feature importance')
    fig.tight_layout(); fig.savefig(PLOT_DIR / f'{name}_feature_importance.png', dpi=160); plt.show()
    return metrics

metrics_table = pd.DataFrame([evaluate_model('random_forest', rf_model), evaluate_model('xgboost', xgb_model)])
metrics_table.to_csv(REPORT_DIR / 'pooled_validation_metrics.csv', index=False)
display(metrics_table)

del X_train, X_val, y_train, y_val
gc.collect()
print('Artifacts saved under:', OUTPUT_ROOT)


## 3. Ablation experiment without U-Net flood fraction

This experiment uses the identical pooled rows, validation mask, numeric preprocessing, model settings, and evaluation outputs from Section 2, but removes only `pred_flood_fraction` from its feature matrix. Its results are isolated from the original experiment.


In [ ]:
ABLATION_DIR = OUTPUT_ROOT / 'without_pred_flood_fraction'
ABLATION_MODEL_DIR = ABLATION_DIR / 'models'
ABLATION_REPORT_DIR = ABLATION_DIR / 'reports'
ABLATION_PLOT_DIR = ABLATION_DIR / 'plots'
for directory in [ABLATION_DIR, ABLATION_MODEL_DIR, ABLATION_REPORT_DIR, ABLATION_PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXCLUDED_FEATURE = 'pred_flood_fraction'
if EXCLUDED_FEATURE not in feature_columns:
    raise ValueError(f'{EXCLUDED_FEATURE!r} is not available in the original feature set.')
ablation_feature_columns = [column for column in feature_columns if column != EXCLUDED_FEATURE]
X_train_ablation = combined.loc[train_mask, ablation_feature_columns].to_numpy(dtype=np.float32, copy=True)
X_val_ablation = combined.loc[val_mask, ablation_feature_columns].to_numpy(dtype=np.float32, copy=True)
y_train_ablation = combined.loc[train_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)
y_val_ablation = combined.loc[val_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)

ablation_signature = pd.DataFrame({'feature': ablation_feature_columns})
ablation_signature.to_csv(ABLATION_REPORT_DIR / 'feature_signature_without_pred_flood_fraction.csv', index=False)
pd.DataFrame([{'experiment': 'without_pred_flood_fraction', 'excluded_feature': EXCLUDED_FEATURE,
               'split_strategy': split_strategy, 'train_rows': len(y_train_ablation),
               'validation_rows': len(y_val_ablation)}]).to_csv(
    ABLATION_REPORT_DIR / 'experiment_metadata.csv', index=False)

ablation_rf_path = ABLATION_MODEL_DIR / 'random_forest.joblib'
ablation_xgb_path = ABLATION_MODEL_DIR / 'xgboost.joblib'
ablation_signature_path = ABLATION_MODEL_DIR / 'feature_signature.csv'
reuse_ablation_models = (ablation_rf_path.exists() and ablation_xgb_path.exists() and
    ablation_signature_path.exists() and pd.read_csv(ablation_signature_path).equals(ablation_signature))

if reuse_ablation_models:
    ablation_rf_model = joblib.load(ablation_rf_path)
    ablation_xgb_model = joblib.load(ablation_xgb_path)
    print('Reused saved no-pred_flood_fraction models.')
else:
    ablation_rf_model = RandomForestClassifier(n_estimators=400, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
    ablation_rf_model.fit(X_train_ablation, y_train_ablation)
    positive_weight = float((y_train_ablation == 0).sum() / max((y_train_ablation == 1).sum(), 1))
    ablation_xgb_model = XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0, objective='binary:logistic',
        eval_metric='logloss', tree_method='hist', max_bin=256, n_jobs=-1, random_state=SEED,
        scale_pos_weight=positive_weight)
    ablation_xgb_model.fit(X_train_ablation, y_train_ablation, verbose=False)
    joblib.dump(ablation_rf_model, ablation_rf_path, compress=3)
    joblib.dump(ablation_xgb_model, ablation_xgb_path, compress=3)
    ablation_signature.to_csv(ablation_signature_path, index=False)

def evaluate_ablation_model(name, model):
    probability = model.predict_proba(X_val_ablation)[:, 1].astype(np.float32)
    prediction = (probability >= 0.5).astype(np.uint8)
    metrics = {'experiment': 'without_pred_flood_fraction', 'model': name,
               'accuracy': accuracy_score(y_val_ablation, prediction),
               'precision': precision_score(y_val_ablation, prediction, zero_division=0),
               'recall': recall_score(y_val_ablation, prediction, zero_division=0),
               'f1': f1_score(y_val_ablation, prediction, zero_division=0),
               'roc_auc': roc_auc_score(y_val_ablation, probability),
               'average_precision': average_precision_score(y_val_ablation, probability)}
    report = classification_report(y_val_ablation, prediction, target_names=['non_flood', 'flood'], output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(ABLATION_REPORT_DIR / f'{name}_classification_report.csv')
    with open(ABLATION_REPORT_DIR / f'{name}_classification_report.txt', 'w') as handle:
        handle.write(classification_report(y_val_ablation, prediction, target_names=['non_flood', 'flood'], zero_division=0))
    id_columns = [c for c in ['state', 'state_id', 'year', 'chunk_id', 'local_patch'] if c in combined]
    predictions = combined.loc[val_mask, id_columns].copy()
    predictions['y_true'], predictions['probability'], predictions['prediction'] = y_val_ablation, probability, prediction
    predictions.to_csv(ABLATION_REPORT_DIR / f'{name}_validation_predictions.csv', index=False)

    cm = confusion_matrix(y_val_ablation, prediction)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, cmap='Blues'); fig.colorbar(image, ax=ax)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['non-flood', 'flood'], yticklabels=['non-flood', 'flood'], xlabel='Predicted', ylabel='True', title=f'{name} without pred_flood_fraction')
    for row in range(2):
        for col in range(2): ax.text(col, row, str(cm[row, col]), ha='center', va='center')
    fig.tight_layout(); fig.savefig(ABLATION_PLOT_DIR / f'{name}_confusion_matrix.png', dpi=160); plt.show()

    fpr, tpr, _ = roc_curve(y_val_ablation, probability)
    precision, recall, _ = precision_recall_curve(y_val_ablation, probability)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.3f}"); axes[0].plot([0, 1], [0, 1], '--', color='gray'); axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'{name} ROC'); axes[0].legend()
    axes[1].plot(recall, precision, label=f"AP={metrics['average_precision']:.3f}"); axes[1].set(xlabel='Recall', ylabel='Precision', title=f'{name} precision-recall'); axes[1].legend()
    fig.tight_layout(); fig.savefig(ABLATION_PLOT_DIR / f'{name}_roc_pr.png', dpi=160); plt.show()

    importance = pd.DataFrame({'feature': ablation_feature_columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(ABLATION_REPORT_DIR / f'{name}_feature_importance.csv', index=False)
    top = importance.head(30).sort_values('importance')
    fig, ax = plt.subplots(figsize=(8, max(5, len(top) * 0.25)))
    ax.barh(top['feature'], top['importance'], color='#2a6f97'); ax.set(xlabel='Importance', title=f'{name} top feature importance')
    fig.tight_layout(); fig.savefig(ABLATION_PLOT_DIR / f'{name}_feature_importance.png', dpi=160); plt.show()
    return metrics

ablation_metrics = pd.DataFrame([
    evaluate_ablation_model('random_forest', ablation_rf_model),
    evaluate_ablation_model('xgboost', ablation_xgb_model),
])
ablation_metrics.to_csv(ABLATION_REPORT_DIR / 'validation_metrics.csv', index=False)
display(ablation_metrics)

del X_train_ablation, X_val_ablation, y_train_ablation, y_val_ablation
gc.collect()
print('No-pred_flood_fraction artifacts saved under:', ABLATION_DIR)


## 4. Cross-state experiment: train Assam and Kerala, validate Odisha

This experiment uses the original full feature set and identical preprocessing, model settings, and evaluation outputs. Only the split changes: all Assam and Kerala rows are training data, while every Odisha row is reserved exclusively for validation.


In [ ]:
CROSS_STATE_DIR = OUTPUT_ROOT / 'cross_state_assam_kerala_train_odisha_validation'
CROSS_STATE_MODEL_DIR = CROSS_STATE_DIR / 'models'
CROSS_STATE_REPORT_DIR = CROSS_STATE_DIR / 'reports'
CROSS_STATE_PLOT_DIR = CROSS_STATE_DIR / 'plots'
for directory in [CROSS_STATE_DIR, CROSS_STATE_MODEL_DIR, CROSS_STATE_REPORT_DIR, CROSS_STATE_PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

cross_state_train_mask = combined['state'].isin(['Assam', 'Kerala']).to_numpy()
cross_state_val_mask = (combined['state'] == 'Odisha').to_numpy()
if not cross_state_train_mask.any() or not cross_state_val_mask.any():
    raise ValueError('Cross-state split requires Assam, Kerala, and Odisha rows in the combined table.')
X_train_cross_state = combined.loc[cross_state_train_mask, feature_columns].to_numpy(dtype=np.float32, copy=True)
X_val_cross_state = combined.loc[cross_state_val_mask, feature_columns].to_numpy(dtype=np.float32, copy=True)
y_train_cross_state = combined.loc[cross_state_train_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)
y_val_cross_state = combined.loc[cross_state_val_mask, TARGET].to_numpy(dtype=np.uint8, copy=True)
if np.unique(y_train_cross_state).size < 2 or np.unique(y_val_cross_state).size < 2:
    raise ValueError('Both cross-state splits must contain flood and non-flood samples.')

cross_state_signature = pd.DataFrame({'feature': feature_columns})
cross_state_signature.to_csv(CROSS_STATE_REPORT_DIR / 'feature_signature.csv', index=False)
cross_state_summary = pd.DataFrame([
    {'split': 'train', 'states': 'Assam,Kerala', 'rows': len(y_train_cross_state), 'flood_rate': y_train_cross_state.mean()},
    {'split': 'validation', 'states': 'Odisha', 'rows': len(y_val_cross_state), 'flood_rate': y_val_cross_state.mean()},
])
cross_state_summary.to_csv(CROSS_STATE_REPORT_DIR / 'cross_state_split_summary.csv', index=False)
display(cross_state_summary)

cross_state_rf_path = CROSS_STATE_MODEL_DIR / 'random_forest.joblib'
cross_state_xgb_path = CROSS_STATE_MODEL_DIR / 'xgboost.joblib'
cross_state_signature_path = CROSS_STATE_MODEL_DIR / 'feature_signature.csv'
reuse_cross_state_models = (cross_state_rf_path.exists() and cross_state_xgb_path.exists() and
    cross_state_signature_path.exists() and pd.read_csv(cross_state_signature_path).equals(cross_state_signature))

if reuse_cross_state_models:
    cross_state_rf_model = joblib.load(cross_state_rf_path)
    cross_state_xgb_model = joblib.load(cross_state_xgb_path)
    print('Reused saved cross-state models.')
else:
    cross_state_rf_model = RandomForestClassifier(n_estimators=400, max_features='sqrt', min_samples_leaf=2,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
    cross_state_rf_model.fit(X_train_cross_state, y_train_cross_state)
    positive_weight = float((y_train_cross_state == 0).sum() / max((y_train_cross_state == 1).sum(), 1))
    cross_state_xgb_model = XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0, objective='binary:logistic',
        eval_metric='logloss', tree_method='hist', max_bin=256, n_jobs=-1, random_state=SEED,
        scale_pos_weight=positive_weight)
    cross_state_xgb_model.fit(X_train_cross_state, y_train_cross_state, verbose=False)
    joblib.dump(cross_state_rf_model, cross_state_rf_path, compress=3)
    joblib.dump(cross_state_xgb_model, cross_state_xgb_path, compress=3)
    cross_state_signature.to_csv(cross_state_signature_path, index=False)

def evaluate_cross_state_model(name, model):
    probability = model.predict_proba(X_val_cross_state)[:, 1].astype(np.float32)
    prediction = (probability >= 0.5).astype(np.uint8)
    metrics = {'experiment': 'cross_state_assam_kerala_to_odisha', 'model': name,
               'accuracy': accuracy_score(y_val_cross_state, prediction),
               'precision': precision_score(y_val_cross_state, prediction, zero_division=0),
               'recall': recall_score(y_val_cross_state, prediction, zero_division=0),
               'f1': f1_score(y_val_cross_state, prediction, zero_division=0),
               'roc_auc': roc_auc_score(y_val_cross_state, probability),
               'average_precision': average_precision_score(y_val_cross_state, probability)}
    report = classification_report(y_val_cross_state, prediction, target_names=['non_flood', 'flood'], output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(CROSS_STATE_REPORT_DIR / f'{name}_classification_report.csv')
    with open(CROSS_STATE_REPORT_DIR / f'{name}_classification_report.txt', 'w') as handle:
        handle.write(classification_report(y_val_cross_state, prediction, target_names=['non_flood', 'flood'], zero_division=0))
    id_columns = [c for c in ['state', 'state_id', 'year', 'chunk_id', 'local_patch'] if c in combined]
    predictions = combined.loc[cross_state_val_mask, id_columns].copy()
    predictions['y_true'], predictions['probability'], predictions['prediction'] = y_val_cross_state, probability, prediction
    predictions.to_csv(CROSS_STATE_REPORT_DIR / f'{name}_odisha_predictions.csv', index=False)

    cm = confusion_matrix(y_val_cross_state, prediction)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, cmap='Blues'); fig.colorbar(image, ax=ax)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['non-flood', 'flood'], yticklabels=['non-flood', 'flood'], xlabel='Predicted', ylabel='True', title=f'{name} Odisha confusion matrix')
    for row in range(2):
        for col in range(2): ax.text(col, row, str(cm[row, col]), ha='center', va='center')
    fig.tight_layout(); fig.savefig(CROSS_STATE_PLOT_DIR / f'{name}_confusion_matrix.png', dpi=160); plt.show()

    fpr, tpr, _ = roc_curve(y_val_cross_state, probability)
    precision, recall, _ = precision_recall_curve(y_val_cross_state, probability)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(fpr, tpr, label=f"AUC={metrics['roc_auc']:.3f}"); axes[0].plot([0, 1], [0, 1], '--', color='gray'); axes[0].set(xlabel='False positive rate', ylabel='True positive rate', title=f'{name} Odisha ROC'); axes[0].legend()
    axes[1].plot(recall, precision, label=f"AP={metrics['average_precision']:.3f}"); axes[1].set(xlabel='Recall', ylabel='Precision', title=f'{name} Odisha precision-recall'); axes[1].legend()
    fig.tight_layout(); fig.savefig(CROSS_STATE_PLOT_DIR / f'{name}_roc_pr.png', dpi=160); plt.show()

    importance = pd.DataFrame({'feature': feature_columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    importance.to_csv(CROSS_STATE_REPORT_DIR / f'{name}_feature_importance.csv', index=False)
    top = importance.head(30).sort_values('importance')
    fig, ax = plt.subplots(figsize=(8, max(5, len(top) * 0.25)))
    ax.barh(top['feature'], top['importance'], color='#2a6f97'); ax.set(xlabel='Importance', title=f'{name} top feature importance')
    fig.tight_layout(); fig.savefig(CROSS_STATE_PLOT_DIR / f'{name}_feature_importance.png', dpi=160); plt.show()
    return metrics

cross_state_metrics = pd.DataFrame([
    evaluate_cross_state_model('random_forest', cross_state_rf_model),
    evaluate_cross_state_model('xgboost', cross_state_xgb_model),
])
cross_state_metrics.to_csv(CROSS_STATE_REPORT_DIR / 'validation_metrics.csv', index=False)
display(cross_state_metrics)

del X_train_cross_state, X_val_cross_state, y_train_cross_state, y_val_cross_state
gc.collect()
print('Cross-state artifacts saved under:', CROSS_STATE_DIR)
